# Study Area Map

Makes the study-area figure for the report: the Sentinel-1 area of interest from `data_collection.ipynb` on a topographic basemap, with the HBRC observed flood extent and a New Zealand inset. Saves `docs/figures/study_area.png` at 300 dpi.

**Before running:**
- Runs locally in the `geog761` environment, not Colab. Paths are relative to this notebook, so run it from `notebooks/`.
- Needs internet: the basemap tiles (Esri World Topographic Map) and the inset outline (Natural Earth 1:10m countries) are downloaded on the fly.
- The flood extent is optional: if `data/reference/` has no HBRC shapefile, the map is drawn without it.
- Place-name coordinates in `PLACES` are approximate; check them against the basemap labels.

In [ ]:
# Cell 1: Install dependencies (contextily downloads the basemap tiles)
%pip install -q geopandas contextily

In [ ]:
import os

import numpy as np
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from shapely.geometry import box

# -------------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------------
# Paths are relative to this notebook (run it from notebooks/)
AOI = (176.60, -39.60, 177.10, -39.00)  # lon_min, lat_min, lon_max, lat_max (as in data_collection.ipynb)
REFERENCE_PATH = os.path.join('..', 'data', 'reference', 'Hawke_s_Bay_Flood_Observation_Extents',
                              'Hawke_s_Bay_Flood_Observation_Extents.shp')
COUNTRIES_URL = 'https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip'
BASEMAP = ctx.providers.Esri.WorldTopoMap
OUTPUT_PATH = os.path.join('..', 'docs', 'figures', 'study_area.png')

# Map extent: a margin around the study area, a little more to the south so Hastings shows
XLIM = (AOI[0] - 0.05, AOI[2] + 0.05)
YLIM = (AOI[1] - 0.08, AOI[3] + 0.04)
PLACES = {  # name: (lon, lat)
    'Napier': (176.912, -39.490),
    'Hastings': (176.842, -39.638),
    'Clive': (176.918, -39.586),
    'Eskdale': (176.833, -39.400),
    'Bay View': (176.865, -39.425),
    'Lake Tūtira': (176.893, -39.222),
}

# Plot in lon/lat so the graticule is exact; stretch the y axis so distances look right at this latitude
lat0 = np.mean([AOI[1], AOI[3]])
km_per_deg_lon = 111.32 * np.cos(np.radians(lat0))


def aoi_rectangle(**kwargs):
    return Rectangle(AOI[:2], AOI[2] - AOI[0], AOI[3] - AOI[1], fill=False, edgecolor='red', **kwargs)

In [ ]:
fig = plt.figure(figsize=(9, 8.5))
grid = fig.add_gridspec(2, 2, width_ratios=[3, 1], height_ratios=[1, 1.4], wspace=0.05, hspace=0.05)

# -------------------------------------------------------------------------
# 2. Main Map
# -------------------------------------------------------------------------
ax = fig.add_subplot(grid[:, 0])
ax.set_xlim(XLIM)
ax.set_ylim(YLIM)
ax.set_aspect(1 / np.cos(np.radians(lat0)))
ctx.add_basemap(ax, crs='EPSG:4326', source=BASEMAP, attribution=False, zoom=11)

handles = []
if os.path.exists(REFERENCE_PATH):
    reference = gpd.read_file(REFERENCE_PATH).to_crs('EPSG:4326').clip(box(*AOI))
    reference.plot(ax=ax, color='tab:blue', alpha=0.55, linewidth=0)
    handles.append(Patch(color='tab:blue', alpha=0.55, label='Observed flood extent,\nFeb 2023 (HBRC)'))

ax.add_patch(aoi_rectangle(linewidth=2))
handles.append(Patch(facecolor='none', edgecolor='red', linewidth=2, label='Study area'))

for name, (lon, lat) in PLACES.items():
    ax.plot(lon, lat, 'o', color='black', markersize=3.5)
    ax.annotate(name, (lon, lat), xytext=(4, 3), textcoords='offset points', fontsize=8.5,
                fontweight='bold' if name in ('Napier', 'Hastings') else 'normal',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.75, linewidth=0))
ax.text(177.04, -39.33, 'Hawke Bay', fontsize=10, fontstyle='italic', color='0.25', ha='center')

# Graticule labels (the basemap call adds CRS axis titles, which are not wanted here)
ax.set_xticks([176.6, 176.8, 177.0])
ax.set_xticklabels([f'{x:.1f}°E' for x in ax.get_xticks()])
ax.set_yticks([-39.6, -39.4, -39.2, -39.0])
ax.set_yticklabels([f'{-y:.1f}°S' for y in ax.get_yticks()])
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(labelsize=8)

# Scale bar (10 km) and north arrow, over the sea
bar_len = 10 / km_per_deg_lon
bar_x0, bar_y = XLIM[1] - 0.04 - bar_len, YLIM[0] + 0.04
ax.add_patch(Rectangle((bar_x0, bar_y), bar_len / 2, 0.006, facecolor='black', edgecolor='black'))
ax.add_patch(Rectangle((bar_x0 + bar_len / 2, bar_y), bar_len / 2, 0.006, facecolor='white', edgecolor='black'))
for frac, label in ((0, '0'), (0.5, '5'), (1, '10 km')):
    ax.text(bar_x0 + frac * bar_len, bar_y + 0.012, label, fontsize=8, ha='center')
ax.annotate('N', xy=(177.05, -39.20), xytext=(177.05, -39.27),
            ha='center', va='center', fontsize=11, fontweight='bold',
            arrowprops=dict(facecolor='black', width=3, headwidth=9, headlength=9))

# -------------------------------------------------------------------------
# 3. Inset: National Context
# -------------------------------------------------------------------------
countries = gpd.read_file(COUNTRIES_URL)
nz = countries[countries['ADMIN'] == 'New Zealand'].clip(box(166, -47.5, 179, -34))

inset = fig.add_subplot(grid[0, 1])
inset.set_facecolor('#dbe9f4')
nz.plot(ax=inset, color='0.85', edgecolor='0.4', linewidth=0.4)
inset.add_patch(aoi_rectangle(linewidth=1.5))
inset.set_xlim(166, 179)
inset.set_ylim(-47.5, -34)
inset.set_aspect(1 / np.cos(np.radians(-41)))
inset.set_xticks([])
inset.set_yticks([])
inset.set_xlabel('')
inset.set_ylabel('')
inset.set_title('New Zealand', fontsize=9)

# -------------------------------------------------------------------------
# 4. Legend and Sources
# -------------------------------------------------------------------------
side = fig.add_subplot(grid[1, 1])
side.axis('off')
side.legend(handles=handles, loc='upper left', fontsize=8, frameon=False)
side.text(0, 0.62, 'Basemap: Esri World\nTopographic Map\nInset: Natural Earth\nFlood extent: Hawke\'s Bay\nRegional Council\nCRS: WGS 84',
          transform=side.transAxes, fontsize=7, color='0.35', va='top', linespacing=1.4)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
plt.savefig(OUTPUT_PATH, dpi=300, bbox_inches='tight')
print(f"Saved '{OUTPUT_PATH}'")
print(f"Basemap attribution: {BASEMAP.attribution}")
plt.show()